In [6]:
# Phase 0 — Kiểm toán dữ liệu Steam

# Mục tiêu:
# - Xác nhận file nguồn, số dòng và số cột.
# - Kiểm tra định danh game.
# - Khảo sát kiểu dữ liệu và giá trị thiếu.
# - Ghi nhận vấn đề trước khi quyết định làm sạch.

# File trong data/raw được giữ nguyên.

In [7]:
from pathlib import Path
import pandas as pd

# Xác định thư mục gốc khi chạy từ repo hoặc notebooks.
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "steam_games.csv"

print("File đang đọc:", DATA_PATH)
print("File tồn tại:", DATA_PATH.is_file())

File đang đọc: d:\Steam Market Intelligence What Makes a Game Succeed on Steam\Steam-Market-Intelligence-What-Makes-a-Game-Succeed-on-Steam\data\raw\steam_games.csv
File tồn tại: True


In [8]:
df = pd.read_csv(DATA_PATH)

print(f"Số dòng: {df.shape[0]:,}")
print(f"Số cột: {df.shape[1]}")

display(df.head())

Số dòng: 71,429
Số cột: 21


,steam_appid,name,developers,publishers,categories,genres,required_age,n_achievements,platforms,is_released,...,additional_content,total_reviews,total_positive,total_negative,review_score,review_score_desc,positive_percentual,metacritic,is_free,price_initial (USD)
0,2719580,勇者の伝説の勇者,['ぽけそう'],['ぽけそう'],"['Single-player', 'Family Sharing']","['Casual', 'Indie']",0,0,['windows'],True,...,[],0,0,0,0.0,No user reviews,0.0,0,False,0.99
1,2719590,Light No Fire,['Hello Games'],['Hello Games'],"['Single-player', 'Multi-player', 'Co-op', 'On...","['Action', 'Adventure']",0,0,['windows'],False,...,[],0,0,0,0.0,No user reviews,0.0,0,False,0.00
2,2719600,Lorhaven: Cursed War,['GoldenGod Games'],['GoldenGod Games'],"['Single-player', 'Multi-player', 'PvP', 'Shar...","['RPG', 'Strategy']",0,32,"['windows', 'mac']",True,...,[],9,8,1,0.0,9 user reviews,88.9,0,False,9.99
3,2719610,PUIQ: Demons,['Giammnn'],['Giammnn'],"['Single-player', 'Steam Achievements', 'Famil...","['Action', 'Casual', 'Indie', 'RPG']",0,28,['windows'],True,...,[],0,0,0,0.0,No user reviews,0.0,0,False,2.99
4,2719650,Project XSTING,['Saucy Melon'],['Saucy Melon'],"['Single-player', 'Steam Achievements', 'Steam...","['Action', 'Casual', 'Indie', 'Early Access']",0,42,['windows'],True,...,[],9,9,0,0.0,9 user reviews,100.0,0,False,7.99


In [9]:
missing_ids = df["steam_appid"].isna().sum()
duplicate_ids = df["steam_appid"].duplicated().sum()

print(f"ID bị thiếu: {missing_ids}")
print(f"ID bị trùng: {duplicate_ids}")

ID bị thiếu: 0
ID bị trùng: 0


In [10]:
audit_summary = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str).to_numpy(),
    "null_count": df.isna().sum().to_numpy(),
    "null_pct": (
        df.isna().mean().mul(100).round(2).to_numpy()
    )
})

display(audit_summary)

,column,dtype,null_count,null_pct
0,steam_appid,int64,0,0.0
1,name,object,0,0.0
2,developers,object,0,0.0
3,publishers,object,0,0.0
4,categories,object,0,0.0
5,genres,object,0,0.0
6,required_age,int64,0,0.0
7,n_achievements,int64,0,0.0
8,platforms,object,0,0.0
9,is_released,bool,0,0.0


In [12]:
## Kiểm tra các trường dạng danh sách

# Mục tiêu: xem cách lưu dữ liệu và đếm danh sách rỗng.
# Chưa thay thế giá trị hoặc xóa dòng.

In [13]:
array_columns = [
    "developers",
    "publishers",
    "categories",
    "genres",
    "platforms",
    "additional_content"
]

# Xem mẫu và kiểu của một giá trị.
display(df[array_columns].head(3))

print("Kiểu giá trị genres:", type(df["genres"].iloc[0]))

,developers,publishers,categories,genres,platforms,additional_content
0,['ぽけそう'],['ぽけそう'],"['Single-player', 'Family Sharing']","['Casual', 'Indie']",['windows'],[]
1,['Hello Games'],['Hello Games'],"['Single-player', 'Multi-player', 'Co-op', 'On...","['Action', 'Adventure']",['windows'],[]
2,['GoldenGod Games'],['GoldenGod Games'],"['Single-player', 'Multi-player', 'PvP', 'Shar...","['RPG', 'Strategy']","['windows', 'mac']",[]


Kiểu giá trị genres: <class 'str'>


In [14]:
array_audit = []

for col in array_columns:
    values = df[col].astype("string").str.strip()

    array_audit.append({
        "column": col,
        "null_count": int(values.isna().sum()),
        "blank_count": int(values.eq("").sum()),
        "empty_list_count": int(values.eq("[]").sum()),
        "empty_list_pct": round(values.eq("[]").mean() * 100, 2)
    })

array_audit = pd.DataFrame(array_audit)

display(array_audit)

,column,null_count,blank_count,empty_list_count,empty_list_pct
0,developers,0,0,200,0.28
1,publishers,0,0,493,0.69
2,categories,0,0,558,0.78
3,genres,0,0,223,0.31
4,platforms,0,0,0,0.00
5,additional_content,0,0,64266,89.97


In [15]:
## Kiểm tra tính nhất quán của review

# Kiểm tra giá trị âm, tổng review và số game không có review.
# Chưa loại game khỏi dữ liệu.

In [18]:
review_columns = [
    "total_reviews",
    "total_positive",
    "total_negative"
]

# Số lượng phải không âm.
negative_counts = (df[review_columns] < 0).sum()
print("Số giá trị âm theo cột:")
print(negative_counts)

# Kiểm tra tổng có khớp hai thành phần không.
calculated_total = df["total_positive"] + df["total_negative"]
mismatch = df["total_reviews"] != calculated_total

print(f"\nGame có tổng review không khớp: {mismatch.sum():,}")

# Khảo sát quy mô mẫu và phân phối.
reviews = df["total_reviews"]

print(f"Game có 0 review: {(reviews == 0).sum():,}")

print(
    f"Game có 1–49 review: "
    f"{((reviews >= 1) & (reviews < 50)).sum():,}"
)

print(f"Game có ít nhất 50 review: {(reviews >= 50).sum():,}")
print(f"Review trung bình: {reviews.mean():,.2f}")
print(f"Review trung vị: {reviews.median():,.2f}")

if mismatch.any():
    display(
        df.loc[
            mismatch,
            ["steam_appid", "name"] + review_columns
        ].head(10)
    )

Số giá trị âm theo cột:
total_reviews     0
total_positive    0
total_negative    0
dtype: int64

Game có tổng review không khớp: 0
Game có 0 review: 26,812
Game có 1–49 review: 33,049
Game có ít nhất 50 review: 11,568
Review trung bình: 410.25
Review trung vị: 2.00


In [19]:
## Review theo trạng thái phát hành

# Phân biệt game đã phát hành và chưa phát hành nhằm xác định
# phạm vi phù hợp cho việc đánh giá mức độ tương tác.
# Tỷ lệ không có review được tính riêng trong từng nhóm.

In [ ]:
release_review_audit = (
    df.groupby("is_released")
    .agg(
        total_games=("steam_appid", "size"),
        zero_review_games=("total_reviews", lambda reviews: (reviews == 0).sum()),
        qualified_games=("total_reviews", lambda reviews: (reviews >= 50).sum()),
        median_reviews=("total_reviews", "median")
    )
)

release_review_audit["zero_review_pct"] = ( release_review_audit["zero_review_games"] / release_review_audit["total_games"] * 100).round(2)
release_review_audit["qualified_pct"] = (release_review_audit["qualified_games"] / release_review_audit["total_games"] * 100).round(2)

display(release_review_audit)

,total_games,zero_review_games,qualified_games,median_reviews,zero_review_pct,qualified_pct
is_released,,,,,,
False,13388,13388,0,0.0,100.00,0.00
True,58041,13424,11568,5.0,23.13,19.93


In [21]:
# Xem mẫu ngày của game đã phát hành.
print("Mẫu release_date của game đã phát hành:")

display(
    df.loc[df["is_released"], "release_date"]
    .drop_duplicates()
    .head(10)
)

# Kiểm tra toàn bộ nhãn ngày ở nhóm chưa phát hành.
print("Giá trị release_date của game chưa phát hành:")

display(
    df.loc[~df["is_released"], "release_date"]
    .value_counts(dropna=False)
)

Mẫu release_date của game đã phát hành:


0     2024-01-04
2     2024-01-26
3     2024-02-17
4     2024-01-05
7     2024-01-15
8     2024-03-26
12    2024-02-29
13    2024-05-23
14    2024-06-08
16    2024-04-06
Name: release_date, dtype: object

Giá trị release_date của game chưa phát hành:


release_date
Not Released    13388
Name: count, dtype: int64

In [26]:
release_dates = df["release_date"].astype("string").str.strip()

# Nhận diện hai cấu trúc đang quan sát được.
date_only = release_dates.str.fullmatch(
    r"\d{4}-\d{2}-\d{2}", na=False
)

date_with_time = release_dates.str.fullmatch(
    r"\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}", na=False
)

# Tạo Series datetime riêng, giữ nguyên cột nguồn.
parsed_dates = pd.Series(pd.NaT, index=df.index, dtype="datetime64[ns]")

parsed_dates.loc[date_only] = pd.to_datetime(
    release_dates.loc[date_only],
    format="%Y-%m-%d",
    errors="coerce"
)

parsed_dates.loc[date_with_time] = pd.to_datetime(
    release_dates.loc[date_with_time],
    format="%Y-%m-%d %H:%M:%S",
    errors="coerce"
)

released_invalid_date = df["is_released"] & parsed_dates.isna()
unreleased_unexpected_label = (~df["is_released"] & release_dates.ne("Not Released"))

print("Số giá trị dạng ngày:", date_only.sum())
print("Số giá trị dạng ngày và giờ:", date_with_time.sum())
print("Game đã phát hành chưa chuyển đổi được ngày:", released_invalid_date.sum())
print("Game chưa phát hành có nhãn khác Not Released:", unreleased_unexpected_label.sum())

valid_dates = parsed_dates.loc[df["is_released"]].dropna()

print("Ngày phát hành sớm nhất:", valid_dates.min())
print("Ngày phát hành muộn nhất:", valid_dates.max())

date_issues = released_invalid_date | unreleased_unexpected_label

if date_issues.any():
    display(
        df.loc[
            date_issues,
            ["steam_appid", "name", "release_date"]
        ].head(10)
    )

Số giá trị dạng ngày: 28
Số giá trị dạng ngày và giờ: 58013
Game đã phát hành chưa chuyển đổi được ngày: 0
Game chưa phát hành có nhãn khác Not Released: 0
Ngày phát hành sớm nhất: 1997-06-30 00:00:00
Ngày phát hành muộn nhất: 2025-01-23 00:00:00
